# Запуск проекта в Google Colab

Этот ноутбук используется как bootstrap-сценарий: его можно загрузить в Colab отдельно от репозитория, после чего он клонирует проект, устанавливает зависимости и запускает эксперименты.

Датасет SportsMOT и веса YOLO11 не хранятся в репозитории. Их нужно заранее получить/загрузить отдельно и указать пути в конфигурации.

## 1. Клонирование репозитория

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/AnixGG/MotFlow.git"
REPO_DIR = Path("/content/MOT")

if not (REPO_DIR / ".git").exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd /content/MOT
!git rev-parse HEAD

## 2. Установка зависимостей

После клонирования репозитория становится доступен `requirements.txt`. Поэтому зависимости устанавливаются из него, чтобы версии совпадали с проектом.

In [ ]:
%cd /content/MOT
!python --version
!nvidia-smi || true
!python -m pip install -U pip setuptools wheel
!python -m pip install -r requirements.txt
!python -m pip install trackeval==1.3.0 --no-deps

## 3. Подключение Google Drive и путь к весам

Веса обученных YOLO11-моделей не хранятся в git. Ниже указывается путь к весам на Google Drive. Если файл лежит в другом месте, измените `YOLO_MODEL_PATH`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

YOLO_MODEL_PATH = Path("/content/drive/MyDrive/pretrained_11yolon.pt")

if YOLO_MODEL_PATH.exists():
    print("Веса найдены:", YOLO_MODEL_PATH)
    !ls -lh {YOLO_MODEL_PATH}
else:
    print("Веса не найдены:", YOLO_MODEL_PATH)
    print("Загрузите файл на Google Drive или измените YOLO_MODEL_PATH.")

## 4. Датасет SportsMOT

SportsMOT требует отдельного доступа/регистрации и не распространяется вместе с этим репозиторием. Для запуска экспериментов validation split должен быть доступен по пути, который указан в конфиге.

Ожидаемый путь по умолчанию:

```text
/content/MOT/dataset/val
```

Внутри должны лежать папки последовательностей с `img1/` и `gt/gt.txt`.

### Опционально: приватная Kaggle-копия

Следующая ячейка оставлена как удобный вариант для автора, если датасет заранее загружен в приватный Kaggle Dataset. Для общего воспроизведения этот блок не обязателен: можно просто смонтировать/загрузить SportsMOT вручную и положить его в `dataset/val`.

По умолчанию блок выключен.

In [ ]:
RUN_PRIVATE_KAGGLE_DOWNLOAD = False
KAGGLE_DATASET = "fot4er/sportsmot"

if RUN_PRIVATE_KAGGLE_DOWNLOAD:
    import json
    import subprocess
    from getpass import getpass
    from pathlib import Path

    username = input("Kaggle username: ")
    key = getpass("Kaggle API key: ")

    kaggle_json = Path("/content/MOT/kaggle.json")
    kaggle_json.write_text(json.dumps({"username": username, "key": key}), encoding="utf-8")

    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(["cp", str(kaggle_json), str(kaggle_dir / "kaggle.json")], check=True)
    subprocess.run(["chmod", "600", str(kaggle_dir / "kaggle.json")], check=True)

    subprocess.run(["python", "-m", "pip", "install", "-q", "kaggle"], check=True)
    subprocess.run(["kaggle", "datasets", "download", KAGGLE_DATASET, "-p", "/content/MOT"], check=True)
    subprocess.run(["unzip", "-q", "-o", "/content/MOT/sportsmot.zip", "-d", "/content/MOT"], check=True)
else:
    print("Приватная Kaggle-загрузка выключена. Подготовьте датасет вручную или включите RUN_PRIVATE_KAGGLE_DOWNLOAD.")

## 5. Проверка данных и обновление путей в конфигах

Эта ячейка проверяет наличие датасета и обновляет поля `model` и `data` в runtime-копиях `configs/run_baseline.yaml` и `configs/run_experiment.yaml`.

In [ ]:
import yaml
from pathlib import Path

REPO_DIR = Path("/content/MOT")
DATA_ROOT = REPO_DIR / "dataset" / "val"
CONFIG_PATHS = [
    REPO_DIR / "configs" / "run_baseline.yaml",
    REPO_DIR / "configs" / "run_experiment.yaml",
]

def to_config_path(path: Path) -> str:
    try:
        return str(path.relative_to(REPO_DIR))
    except ValueError:
        return str(path)

if DATA_ROOT.exists():
    sequences = sorted(
        item.name for item in DATA_ROOT.iterdir()
        if (item / "img1").exists() and (item / "gt" / "gt.txt").exists()
    )
    print("Найдено sequences:", len(sequences))
    print("Первые sequences:", sequences[:5])
else:
    print("Датасет не найден:", DATA_ROOT)
    print("Проверьте путь DATA_ROOT или распакуйте SportsMOT в /content/MOT/dataset/val.")

for config_path in CONFIG_PATHS:
    config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
    config["model"] = str(YOLO_MODEL_PATH)
    config["data"] = to_config_path(DATA_ROOT)
    config_path.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding="utf-8")
    print("Обновлен конфиг:", config_path)

## 6. Проверка GPU

In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 7. Запуск экспериментов

In [ ]:
%cd /content/MOT
!python -B scripts/run_baseline.py --config configs/run_baseline.yaml

In [ ]:
%cd /content/MOT
!python -B scripts/run_experiment.py --config configs/run_experiment.yaml

## 8. Сохранение результатов на Google Drive

Результаты копируются из директорий `outdir`, указанных в YAML-конфигах.

In [ ]:
import shutil
import yaml
from pathlib import Path

REPO_DIR = Path("/content/MOT")
DRIVE_EXPORT = Path("/content/drive/MyDrive/MOT_experiments")
DRIVE_EXPORT.mkdir(parents=True, exist_ok=True)

for config_path in [REPO_DIR / "configs" / "run_baseline.yaml", REPO_DIR / "configs" / "run_experiment.yaml"]:
    config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
    outdir = Path(str(config["outdir"]))
    if not outdir.is_absolute():
        outdir = REPO_DIR / outdir

    if not outdir.exists():
        print("Директория результата не найдена, пропускаю:", outdir)
        continue

    target = DRIVE_EXPORT / outdir.name
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(outdir, target)
    print("Скопировано:", outdir, "->", target)